# LumenY 5.1 — 01: Data Pipeline (Cross Pairs)

Downloads historical 1-min OHLCV data for **8 new FX cross pairs** from Polygon/Massive S3.

**New Pairs:** EURJPY, GBPJPY, EURGBP, EURAUD, AUDJPY, CADJPY, CHFJPY, AUDNZD

**Timeframes saved:** 1min (raw), 5m, 15m, 1H, 4H, 1D, 1W

**Output:** Raw parquet in `backend/data/raw/`, processed in `backend/data/processed/`

> Exact replica of `notebooks/01_data_pipeline.ipynb` but for cross pairs only.

In [1]:
import os
import gzip
import boto3
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from botocore.config import Config
from tqdm import tqdm
from io import BytesIO

load_dotenv('../.env')

# Paths
RAW_DIR       = Path('../backend/data/raw')
PROCESSED_DIR = Path('../backend/data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Directories ready.')
print(f'Raw:       {RAW_DIR.resolve()}')
print(f'Processed: {PROCESSED_DIR.resolve()}')

Directories ready.
Raw:       C:\Users\noual\lumeny\backend\data\raw
Processed: C:\Users\noual\lumeny\backend\data\processed


## 1. Connect to S3

In [2]:
ACCESS_KEY = os.getenv('POLYGON_S3_ACCESS_KEY')
SECRET_KEY = os.getenv('POLYGON_S3_SECRET_KEY')

if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError('Missing S3 credentials in .env — check POLYGON_S3_ACCESS_KEY and POLYGON_S3_SECRET_KEY')

session = boto3.Session(
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)

s3 = session.client(
    's3',
    endpoint_url='https://files.massive.com',
    config=Config(signature_version='s3v4'),
)

BUCKET = 'flatfiles'
PREFIX = 'global_forex/minute_aggs_v1'

print('S3 client ready.')

S3 client ready.


## 2. Define Download Parameters

In [3]:
# Cross pairs — Polygon uses C:EUR-JPY format internally
PAIRS = ['EURJPY', 'GBPJPY', 'EURGBP', 'EURAUD', 'AUDJPY', 'CADJPY', 'CHFJPY', 'AUDNZD']

# Download range
START_YEAR = 2009
END_YEAR   = 2025

print(f'Pairs:  {PAIRS}')
print(f'Range:  {START_YEAR} - {END_YEAR}')

Pairs:  ['EURJPY', 'GBPJPY', 'EURGBP', 'EURAUD', 'AUDJPY', 'CADJPY', 'CHFJPY', 'AUDNZD']
Range:  2009 - 2025


## 3. Download Raw Minute Data

Files are organized as daily `.csv.gz` files. We download, decompress, filter for our pairs, and save.

In [4]:
def list_files_for_year(year: int) -> list:
    """List all daily files available for a given year."""
    prefix = f'{PREFIX}/{year}/'
    paginator = s3.get_paginator('list_objects_v2')
    files = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get('Contents', []):
            files.append(obj['Key'])
    return sorted(files)


def download_and_parse_file(key: str, pairs: list) -> pd.DataFrame:
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=key)
        compressed = obj['Body'].read()
        
        with gzip.open(BytesIO(compressed), 'rt') as f:
            df = pd.read_csv(f)
        
        df.columns = [c.lower().strip() for c in df.columns]
        
        # Polygon forex format: C:EUR-USD — normalize to EURUSD
        df['pair'] = df['ticker'].str.replace('C:', '', regex=False).str.replace('-', '', regex=False)
        
        # Filter for our pairs
        df = df[df['pair'].isin(pairs)]
        
        if len(df) == 0:
            return None
        
        # window_start is nanoseconds — convert to datetime
        df['datetime'] = pd.to_datetime(df['window_start'], unit='ns')
        
        # Keep only what we need
        df = df[['datetime', 'pair', 'open', 'high', 'low', 'close', 'volume']].dropna()
        
        return df
    
    except Exception as e:
        print(f'  ERROR reading {key}: {e}')
        return None

print('Download functions ready.')

Download functions ready.


In [5]:
# Download all data year by year
# Each year is saved as a separate parquet per pair to keep memory manageable

for year in range(START_YEAR, END_YEAR + 1):
    print(f'\n--- Year {year} ---')
    
    files = list_files_for_year(year)
    print(f'  Found {len(files)} daily files')
    
    if not files:
        print(f'  No files found for {year}, skipping.')
        continue
    
    # Accumulate data for the year
    year_data = {pair: [] for pair in PAIRS}
    
    for key in tqdm(files, desc=f'{year}'):
        df = download_and_parse_file(key, PAIRS)
        if df is None or len(df) == 0:
            continue
        
        for pair in PAIRS:
            pair_df = df[df['pair'] == pair]
            if len(pair_df) > 0:
                year_data[pair].append(pair_df)
    
    # Save each pair's yearly data
    for pair in PAIRS:
        if not year_data[pair]:
            print(f'  No data for {pair} in {year}')
            continue
        
        df_pair = pd.concat(year_data[pair], ignore_index=True)
        df_pair = df_pair.sort_values('datetime').drop_duplicates(subset=['datetime'])
        
        out_path = RAW_DIR / f'{pair}_1min_{year}.parquet'
        df_pair.to_parquet(out_path, index=False)
        print(f'  {pair} {year}: {len(df_pair)} rows saved')

print('\nAll downloads complete!')


--- Year 2009 ---
  Found 84 daily files


2009: 100%|██████████| 84/84 [03:22<00:00,  2.41s/it]


  EURJPY 2009: 96828 rows saved
  GBPJPY 2009: 97004 rows saved
  EURGBP 2009: 96515 rows saved
  EURAUD 2009: 97006 rows saved
  AUDJPY 2009: 96931 rows saved
  CADJPY 2009: 95927 rows saved
  CHFJPY 2009: 96924 rows saved
  AUDNZD 2009: 95845 rows saved

--- Year 2010 ---
  Found 314 daily files


2010: 100%|██████████| 314/314 [13:45<00:00,  2.63s/it]


  EURJPY 2010: 367748 rows saved
  GBPJPY 2010: 366442 rows saved
  EURGBP 2010: 366647 rows saved
  EURAUD 2010: 367278 rows saved
  AUDJPY 2010: 366139 rows saved
  CADJPY 2010: 364644 rows saved
  CHFJPY 2010: 367386 rows saved
  AUDNZD 2010: 351837 rows saved

--- Year 2011 ---
  Found 312 daily files


...

  EURJPY 2023: 383047 rows saved
  GBPJPY 2023: 380633 rows saved
  EURGBP 2023: 370770 rows saved
  EURAUD 2023: 372379 rows saved
  AUDJPY 2023: 376010 rows saved
  CADJPY 2023: 390143 rows saved
  CHFJPY 2023: 376262 rows saved
  AUDNZD 2023: 502801 rows saved

--- Year 2024 ---
  Found 314 daily files


2024: 100%|██████████| 314/314 [09:22<00:00,  1.79s/it]


  EURJPY 2024: 368603 rows saved
  GBPJPY 2024: 369159 rows saved
  EURGBP 2024: 370654 rows saved
  EURAUD 2024: 370824 rows saved
  AUDJPY 2024: 372565 rows saved
  CADJPY 2024: 372330 rows saved
  CHFJPY 2024: 370822 rows saved
  AUDNZD 2024: 372585 rows saved

--- Year 2025 ---
  Found 312 daily files


2025: 100%|██████████| 312/312 [11:10<00:00,  2.15s/it]


  EURJPY 2025: 363585 rows saved
  GBPJPY 2025: 362770 rows saved
  EURGBP 2025: 370539 rows saved
  EURAUD 2025: 370555 rows saved
  AUDJPY 2025: 370601 rows saved
  CADJPY 2025: 370594 rows saved
  CHFJPY 2025: 370583 rows saved
  AUDNZD 2025: 370608 rows saved

All downloads complete!


## 4. Combine Years and Resample to All Timeframes

In [6]:
def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """Resample OHLCV DataFrame to a given pandas rule."""
    resampled = df.resample(rule).agg({
        'open':   'first',
        'high':   'max',
        'low':    'min',
        'close':  'last',
        'volume': 'sum'
    }).dropna()
    # Remove weekends
    resampled = resampled[resampled.index.dayofweek < 5]
    return resampled


# Timeframes to generate
TIMEFRAMES = {
    '5m':  '5min',
    '15m': '15min',
    '1H':  '1h',
    '4H':  '4h',
    '1D':  '1D',
    '1W':  '1W',
}

print('Resample function ready.')
print(f'Timeframes to generate: {list(TIMEFRAMES.keys())}')

Resample function ready.
Timeframes to generate: ['5m', '15m', '1H', '4H', '1D', '1W']


In [7]:
for pair in PAIRS:
    print(f'\nProcessing {pair}...')
    
    # Collect all yearly files for this pair
    yearly_files = sorted(RAW_DIR.glob(f'{pair}_1min_*.parquet'))
    
    if not yearly_files:
        print(f'  No raw files found, skipping.')
        continue
    
    # Combine all years
    dfs = []
    for f in yearly_files:
        df = pd.read_parquet(f)
        dfs.append(df)
    
    df_1min = pd.concat(dfs, ignore_index=True)
    df_1min = df_1min.sort_values('datetime').drop_duplicates(subset=['datetime'])
    df_1min = df_1min.set_index('datetime')
    
    # Remove timezone if present
    if df_1min.index.tz is not None:
        df_1min.index = df_1min.index.tz_localize(None)
    
    # Keep only OHLCV
    df_1min = df_1min[['open', 'high', 'low', 'close', 'volume']]
    df_1min = df_1min[df_1min.index.dayofweek < 5]
    
    print(f'  1min combined: {len(df_1min)} rows | {df_1min.index[0].date()} -> {df_1min.index[-1].date()}')
    
    # Resample and save each timeframe
    for tf_name, tf_rule in TIMEFRAMES.items():
        try:
            df_tf = resample_ohlcv(df_1min, tf_rule)
            out_path = PROCESSED_DIR / f'{pair}_{tf_name}.parquet'
            df_tf.to_parquet(out_path)
            print(f'  {tf_name}: {len(df_tf)} candles saved')
        except Exception as e:
            print(f'  ERROR resampling {tf_name}: {e}')

print('\nAll pairs processed!')


Processing EURJPY...
  1min combined: 5892143 rows | 2009-09-25 -> 2025-12-31
  5m: 1179990 candles saved
  15m: 393593 candles saved
  1H: 98594 candles saved
  4H: 25155 candles saved
  1D: 4210 candles saved
  1W: 0 candles saved

Processing GBPJPY...
  1min combined: 5878738 rows | 2009-09-25 -> 2025-12-31
  5m: 1177529 candles saved
  15m: 392772 candles saved
  1H: 98388 candles saved
  4H: 25106 candles saved
  1D: 4204 candles saved
  1W: 0 candles saved

Processing EURGBP...
  1min combined: 5865910 rows | 2009-09-25 -> 2025-12-31
  5m: 1176884 candles saved
  15m: 392690 candles saved
  1H: 98473 candles saved
  4H: 25172 candles saved
  1D: 4211 candles saved
  1W: 0 candles saved

Processing EURAUD...
  1min combined: 5870615 rows | 2009-09-25 -> 2025-12-31
  5m: 1176063 candles saved
  15m: 392370 candles saved
  1H: 98357 candles saved
  4H: 25128 candles saved
  1D: 4204 candles saved
  1W: 0 candles saved

Processing AUDJPY...
  1min combined: 5908572 rows | 2009-09-25

## 5. Validate

In [8]:
# Quick check on one pair
pair_to_check = 'EURJPY'
tf_to_check   = '1H'

df_check = pd.read_parquet(PROCESSED_DIR / f'{pair_to_check}_{tf_to_check}.parquet')

print(f'Shape:      {df_check.shape}')
print(f'Date range: {df_check.index[0]} -> {df_check.index[-1]}')
print(f'Columns:    {df_check.columns.tolist()}')
print(f'\nNull values:\n{df_check.isnull().sum()}')
print(f'\nLast 5 rows:')
df_check.tail(5)

Shape:      (98594, 5)
Date range: 2009-09-25 04:00:00 -> 2025-12-31 22:00:00
Columns:    ['open', 'high', 'low', 'close', 'volume']

Null values:
open      0
high      0
low       0
close     0
volume    0
dtype: int64

Last 5 rows:


,open,high,low,close,volume
datetime,,,,,
2025-12-31 15:00:00,184.081,184.165,183.940,184.082,17712
2025-12-31 16:00:00,184.081,184.127,183.907,184.015,15660
2025-12-31 17:00:00,184.025,184.060,183.940,184.052,11749
2025-12-31 18:00:00,184.054,184.075,184.028,184.054,1497
2025-12-31 22:00:00,184.054,184.054,184.054,184.054,1


In [10]:
# Full summary of all new pairs
print('Processed files summary:\n')
print(f'{"Pair":<10} {"TF":<6} {"Candles":<10} {"From":<14} {"To"}')
print('-' * 58)

for pair in PAIRS:
    for tf in ['5m', '15m', '1H', '4H', '1D', '1W']:
        path = PROCESSED_DIR / f'{pair}_{tf}.parquet'
        if path.exists():
            df = pd.read_parquet(path)
            if len(df) > 0:
                print(f'{pair:<10} {tf:<6} {len(df):<10} {str(df.index[0].date()):<14} {df.index[-1].date()}')
            else:
                print(f'{pair:<10} {tf:<6} EMPTY')
        else:
            print(f'{pair:<10} {tf:<6} MISSING')

Processed files summary:

Pair       TF     Candles    From           To
----------------------------------------------------------
EURJPY     5m     1179990    2009-09-25     2025-12-31
EURJPY     15m    393593     2009-09-25     2025-12-31
EURJPY     1H     98594      2009-09-25     2025-12-31
EURJPY     4H     25155      2009-09-25     2025-12-31
EURJPY     1D     4210       2009-09-25     2025-12-31
EURJPY     1W     EMPTY
GBPJPY     5m     1177529    2009-09-25     2025-12-31
GBPJPY     15m    392772     2009-09-25     2025-12-31
GBPJPY     1H     98388      2009-09-25     2025-12-31
GBPJPY     4H     25106      2009-09-25     2025-12-31
GBPJPY     1D     4204       2009-09-25     2025-12-31
GBPJPY     1W     EMPTY
EURGBP     5m     1176884    2009-09-25     2025-12-31
EURGBP     15m    392690     2009-09-25     2025-12-31
EURGBP     1H     98473      2009-09-25     2025-12-31
EURGBP     4H     25172      2009-09-25     2025-12-31
EURGBP     1D     4211       2009-09-25     2025-1